# Parte 2 — Operação

**Banco Aurora · Prevenção a Fraudes** · entrega: **sexta-feira, 13/11, 23h59**

Leia antes o `demanda_parte2.md`, só o e-mail principal. Os anexos têm hora para serem lidos.

Na parte 1 você colocou um modelo em produção e preparou o monitoramento. Agora você opera esse modelo: quatro semanas chegam, uma por vez, e a cada semana você decide o que fazer. A decisão pode ser "nada".

**Como este notebook funciona**
- Ele segue o roteiro da demanda: uma seção por semana, na ordem.
- Três seções ficam no fim: **I**nvestigação, **D**ecisão e **A**ção. Use-as quando, e se, o monitoramento ou o negócio pedirem. Você pode voltar a elas em qualquer semana.
- O sistema é operado pelas **telas**: Lotes, Rótulos, Decisões, Modelo e Painel. Se quiser automatizar alguma coisa a partir do notebook, as rotas da API estão em `http://127.0.0.1:8000/docs`. Nesse caso, o código é seu.
- Duas células vêm prontas: a de ambiente e a de entrega.
- O código continua sendo seu. O que você reaproveitar na parte 3 vai para `features.py` e `monitoramento.py`.

**O que será avaliado nesta parte** (a nota descreve o **estado** do que você entregou)
1. O modelo em produção respeita o contrato de features, no treino e na inferência.
2. O que está em produção é rastreável: MLflow, contrato e `CHANGELOG.md` contam a mesma história.
3. O monitoramento existia e foi usado **toda semana**, antes do diagnóstico.
4. O diagnóstico do que aconteceu está correto.
5. A ação é proporcional ao problema: retreinar só quando necessário.
6. A comunicação com o negócio é clara, em reais e em clientes, sem jargão.

## 0. Ambiente

Rode a célula abaixo. Ela confere as conexões e lista as versões do `antifraude` que existem no seu MLflow.

In [1]:
# CÉLULA PRONTA — só rode. Tudo precisa aparecer como "ok".
import json
import urllib.request

import mlflow
import psycopg2
from mlflow.tracking import MlflowClient

import config

for nome, uri in [("Postgres / antifraude", config.URI_BANCO_APP), ("Postgres / mlflow", config.URI_BANCO_MLFLOW)]:
    try:
        psycopg2.connect(uri).close()
        print(f"{nome:24} ok")
    except Exception as e:
        print(f"{nome:24} FALHOU -> {e}")

for nome, url in [("MLflow", f"{config.MLFLOW_TRACKING_URI}/health"), ("Sistema", f"{config.SISTEMA_URL}/saude")]:
    try:
        resp = urllib.request.urlopen(url, timeout=5).read().decode()
        print(f"{nome:24} ok -> {resp[:160]}")
    except Exception as e:
        print(f"{nome:24} FALHOU -> {e}")

mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
cli = MlflowClient()
print("\nversões de", config.NOME_MODELO, "no MLflow:")
for v in sorted(cli.search_model_versions(f"name='{config.NOME_MODELO}'"), key=lambda v: int(v.version)):
    alias = ", ".join("@" + a for a in v.aliases)
    print(f"  v{v.version:<3} {alias:12} threshold={v.tags.get(config.TAG_THRESHOLD, '—'):6} {v.tags.get(config.TAG_MOTIVO, '')}")

Postgres / antifraude    ok
Postgres / mlflow        ok
MLflow                   ok -> OK


Sistema                  ok -> {"sistema":"ok","contrato":"1.0","modelo":{"ok":true,"versao":"7","threshold":0.09,"run_id":"98e2f8417abf497ebb1132c0a39f5258","descricao":"Algoritmo: floresta_

versões de antifraude no MLflow:


  v1                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v2                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v3                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v4                threshold=0.29   v1.0.0 - primeiro modelo em produção
  v5                threshold=0.09   Menor custo na comparação ampliada das semanas 11 e 12
  v6                threshold=0.09   Menor custo na comparação ampliada das semanas 11 e 12
  v7                threshold=0.09   Menor custo na comparação ampliada das semanas 11 e 12


## 1. Antes do primeiro lote

### 1.1 Qual modelo vai operar?
Confira o seu `@champion`. Ele tem a tag `threshold`? A run dele tem a `referencia/`? O sistema pontua com ele (veja a tela **Modelo**)?

**Você escolhe com qual modelo vai operar:**
- **Com o seu modelo da parte 1.** Ele precisa estar funcionando: em produção, com threshold e com referência.
- **Com o baseline oficial** (`baseline_oficial/`). Vale se você não conseguiu chegar a um bom modelo na parte 1, se algo ficou quebrado, ou se prefere partir de uma base conhecida. Usar o baseline **não tira ponto** nesta parte. O passo a passo está no `baseline_oficial/LEIAME.md`.

Registre a escolha e o motivo na decisão da semana 13. Se trocar o champion para o baseline, isso entra também no `CHANGELOG.md`.

### 1.2 O relatório da semana
Você vai rodar o mesmo relatório quatro vezes. Escreva-o uma vez só, de preferência como uma função no `monitoramento.py` (ex.: `relatorio_semana(n)`), que:
- carrega a **referência** do champion. Baixe-a do MLflow (`mlflow.artifacts.download_artifacts(run_id=..., artifact_path="referencia")`), não de uma cópia solta: trocou o champion, trocou a referência;
- lê os dados pontuados da semana: o CSV do link *baixar dados* da tela **Lotes**, ou a rota `GET /api/semanas/{n}/dados.csv`;
- devolve as tabelas de drift, a saída do modelo e a performance da última semana com rótulo, usando os limites de alerta que você definiu na parte 1.

> Teste o relatório antes de enviar o primeiro lote. Use a própria referência contra ela mesma, ou a semana 11 contra a 12, como no relatório zero.

Os relatórios abaixo mostram os CSVs arquivados no momento de cada rodada. Para as semanas 15 e 16, há também tabelas após o ajuste de horário. A função `relatorio_semana` em `monitoramento.py` pode consultar a API e a referência no MLflow novamente.

In [2]:
from IPython.display import display

from monitoramento import carregar_referencia_champion, monitorar, relatorio_semana

referencia, corte, versao = carregar_referencia_champion()
with urllib.request.urlopen(f"{config.SISTEMA_URL}/api/saude", timeout=15) as resposta:
    producao = json.load(resposta)["modelo"]
assert producao["ok"], producao
assert str(producao["versao"]) == versao and producao["threshold"] == corte, producao
print(f"Champion v{versao} | corte {corte} | referência com {len(referencia):,} transações")

Champion v7 | corte 0.09 | referência com 39,497 transações


In [3]:
controle = monitorar(referencia, referencia, corte)
assert all(tabela is not None and not tabela["alerta"].any() for tabela in controle.values())
display(controle["saida"])

,conjunto,transacoes,bloqueios,taxa_bloqueio,valor_bloqueado,valor_total,pct_valor_bloqueado,delta_taxa_bloqueio_pp,delta_pct_valor_bloqueado_pp,alerta
0,referencia,39497,1095,0.027724,603268.81,7334564.93,0.08225,0.0,0.0,False
1,lote,39497,1095,0.027724,603268.81,7334564.93,0.08225,0.0,0.0,False


In [4]:
from pathlib import Path

import pandas as pd

pasta_relatorios = Path("monitoramento")
if not (pasta_relatorios / "semana_13").is_dir():
    pasta_relatorios = Path("evidencias_operacao")
pasta_exportacao = Path("entrega_parte2")
if not (pasta_exportacao / "decisoes.csv").is_file():
    pasta_exportacao = Path.cwd()

def mostrar_relatorio(semana, *, apos_correcao=False):
    pasta = pasta_relatorios / f"semana_{semana}"
    if apos_correcao:
        pasta = pasta / "apos_correcao"
    tabelas = {}
    for nome in ("continuas", "discretas", "saida", "performance"):
        arquivo = pasta / f"{nome}.csv"
        if not arquivo.is_file() and semana == 13 and nome == "performance":
            print("Na semana 13, as primeiras contestações ainda não tinham chegado.")
            continue
        tabela = pd.read_csv(arquivo)
        tabelas[nome] = tabela
        print(nome)
        display(tabela)
    return tabelas

def mostrar_decisao(semana):
    decisoes = pd.read_csv(pasta_exportacao / "decisoes.csv")
    registro = decisoes.loc[decisoes["semana"].eq(semana)]
    assert len(registro) == 1, f"Esperava uma decisão registrada para a semana {semana}"
    display(registro[["semana", "versao_modelo", "versao_contrato", "retreinou", "acao"]])

*Qual modelo vai operar e por quê:*

A operação começou com a Random Forest `antifraude v7`, corte 0,09 e contrato v1.0. O alias, o corte e a referência de 39.497 transações foram conferidos no MLflow, e a API pontuou com essa versão. Nas semanas 11 e 12, a v7 custou R$ 18.866,83. O pacote informa R$ 28.448,23 para o baseline oficial.

## Semana 13

**Chegam:** `dados/parte2/transacoes_semana_13.csv`.

1. Envie o lote na tela **Lotes** do sistema.
   Os dados já pontuados da semana saem na própria tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 13 contra a referência (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - a performance ainda não dá para medir: nenhum rótulo novo chegou;
3. Leia o relatório e escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 13 na tela **Decisões** do sistema.

Se você decidir agir agora, faça a investigação e a ação (seções I, D e A, no fim do notebook) **antes** de seguir para a próxima semana. Registre a decisão desta semana com o que você fez.

> **Antes da semana 14:** leia o **Anexo 1** da demanda.

In [5]:
relatorio_13 = mostrar_relatorio(13)

continuas


,variavel,n_referencia,n_lote,invalidos_referencia,invalidos_lote,status,media_referencia,media_lote,D,p_valor,alerta
0,valor_transacao,39497,19419,0,0,calculado,185.699292,184.310755,0.006517,6.354666e-01,False
1,qtd_transacoes_1h,39497,19419,0,0,calculado,0.365445,0.368711,0.003719,9.934721e-01,False
2,valor_medio_30d,39497,19419,0,0,calculado,118.092749,117.163250,0.005727,7.843286e-01,False
3,razao_valor_media,39497,19419,0,0,calculado,1.581537,1.580414,0.010073,1.414301e-01,False
4,distancia_ultima_transacao_km,39497,19419,0,0,calculado,22.099109,22.945090,0.007601,4.374021e-01,False
5,idade_conta_dias,39497,19419,0,0,calculado,816.248196,827.582831,0.061383,4.400832e-43,False
6,score,39497,19419,0,0,calculado,0.014888,0.014610,0.008382,3.180045e-01,False


discretas


,variavel,n_referencia,n_lote,categorias,categoria_maior_mudanca,maior_mudanca_pp,p_valor,alerta
0,hora_transacao,39497,19419,24,7,0.457379,0.065836,False
1,dia_semana,39497,19419,7,3,0.638378,0.163921,False
2,canal,39497,19419,3,cartao_online,0.730459,0.116744,False
3,dispositivo_novo,39497,19419,2,1,0.098846,0.565882,False
4,destinatario_novo,39497,19419,2,1,0.487014,0.189819,False
5,categoria_estabelecimento,39497,19419,10,transferencia_pessoal,0.591644,0.411607,False
6,uf_cliente,39497,19419,27,RS,0.391767,0.092474,False


saida


,conjunto,transacoes,bloqueios,taxa_bloqueio,valor_bloqueado,valor_total,pct_valor_bloqueado,delta_taxa_bloqueio_pp,delta_pct_valor_bloqueado_pp,alerta,semana,versao_modelo,threshold
0,referencia,39497,1095,0.027724,603268.81,7334564.93,0.082250,0.000000,0.00000,False,NaN,7,0.09
1,lote,19419,515,0.026520,300237.57,3579130.56,0.083886,-0.120321,0.16355,False,13.0,7,0.09


Na semana 13, as primeiras contestações ainda não tinham chegado.


In [6]:
assert not any(tabela["alerta"].any() for tabela in relatorio_13.values())
assert "performance" not in relatorio_13
mostrar_decisao(13)

,semana,versao_modelo,versao_contrato,retreinou,acao
0,13,7,1.0,False,"Manter a Random Forest v7 como champion, com c..."


*Semana 13 — o que eu vi e o que vou fazer:*

O sistema pontuou 19.419 transações. Encaminhou 515 para verificação (2,65%, ante 2,77% na validação), somando R$ 300.237,57 em valor retido. As entradas, os scores e a saída ficaram dentro dos limites de alerta. A decisão foi registrada no sistema: manter a v7 com corte 0,09 e medir o custo e os bloqueios indevidos quando chegarem as contestações da semana 13.

## Semana 14

**Chegam:** `dados/parte2/transacoes_semana_14.csv` e `dados/parte2/rotulos_semana_13.csv`.

1. Envie o lote na tela **Lotes** do sistema e, **depois**, os rótulos na tela **Rótulos**.
   Os dados já pontuados da semana saem na própria tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 14 contra a referência (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - **performance da semana 13**, que acabou de ganhar rótulo: recall, precision, PR-AUC e custo, globais **e por segmento**; compare com a referência e com as semanas anteriores;
3. Leia o relatório e escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 14 na tela **Decisões** do sistema.

Se você decidir agir agora, faça a investigação e a ação (seções I, D e A, no fim do notebook) **antes** de seguir para a próxima semana. Registre a decisão desta semana com o que você fez.

In [7]:
relatorio_14 = mostrar_relatorio(14)

continuas


,variavel,n_referencia,n_lote,invalidos_referencia,invalidos_lote,status,media_referencia,media_lote,D,p_valor,alerta
0,valor_transacao,39497,20483,0,0,calculado,185.699292,185.198859,0.008901,2.341690e-01,False
1,qtd_transacoes_1h,39497,20483,0,0,calculado,0.365445,0.367866,0.003897,9.861232e-01,False
2,valor_medio_30d,39497,20483,0,0,calculado,118.092749,117.882049,0.008339,3.037120e-01,False
3,razao_valor_media,39497,20483,0,0,calculado,1.581537,1.578829,0.009284,1.941470e-01,False
4,distancia_ultima_transacao_km,39497,20483,0,0,calculado,22.099109,22.651633,0.007550,4.233466e-01,False
5,idade_conta_dias,39497,20483,0,0,calculado,816.248196,836.119953,0.060245,5.354909e-43,False
6,score,39497,20483,0,0,calculado,0.014888,0.018806,0.078819,2.488462e-73,False


discretas


,variavel,n_referencia,n_lote,categorias,categoria_maior_mudanca,maior_mudanca_pp,p_valor,alerta
0,hora_transacao,39497,20483,24,23,4.563415,0.000000,False
1,dia_semana,39497,20483,7,3,0.607988,0.412891,False
2,canal,39497,20483,3,cartao_presencial,0.639348,0.259881,False
3,dispositivo_novo,39497,20483,2,1,0.209587,0.203068,False
4,destinatario_novo,39497,20483,2,0,0.082256,0.827749,False
5,categoria_estabelecimento,39497,20483,10,restaurante,0.676210,0.283130,False
6,uf_cliente,39497,20483,27,SP,0.367020,0.771925,False


saida


,conjunto,transacoes,bloqueios,taxa_bloqueio,valor_bloqueado,valor_total,pct_valor_bloqueado,delta_taxa_bloqueio_pp,delta_pct_valor_bloqueado_pp,alerta,semana,versao_modelo,threshold
0,referencia,39497,1095,0.027724,603268.81,7334564.93,0.082250,0.000000,0.000000,False,NaN,7,0.09
1,lote,19419,515,0.026520,300237.57,3579130.56,0.083886,-0.120321,0.163550,False,13.0,7,0.09
2,lote,20483,812,0.039643,391360.35,3793428.23,0.103168,1.191901,2.091787,True,14.0,7,0.09


performance


,conjunto,segmento,valor_segmento,transacoes,fraudes,rotuladas,cobertura_rotulos,recall,precision,pr_auc,custo_observado,custo,custo_por_mil,motivos_alerta,alerta,status_comparacao,semana,versao_modelo,threshold
0,referencia,global,todos,39497,600,39497,1.0,0.851667,0.466667,0.818789,18866.83,18866.83,477.677545,NaN,False,referencia,NaN,7,0.09
1,referencia,canal,cartao_online,9871,201,9871,1.0,0.905473,0.434368,0.860174,5858.16,5858.16,593.471786,NaN,False,referencia,NaN,7,0.09
2,referencia,canal,cartao_presencial,11828,113,11828,1.0,0.761062,0.352459,0.630069,5731.43,5731.43,484.564592,NaN,False,referencia,NaN,7,0.09
3,referencia,canal,pix,17798,286,17798,1.0,0.849650,0.562500,0.849862,7277.24,7277.24,408.879649,NaN,False,referencia,NaN,7,0.09
4,lote,global,todos,19419,284,19419,1.0,0.848592,0.467961,0.838318,10986.30,10986.30,565.750039,NaN,False,comparado,13.0,7,0.09
5,lote,canal,cartao_online,4995,122,4995,1.0,0.868852,0.477477,0.854422,4798.80,4798.80,960.720721,NaN,False,comparado,13.0,7,0.09
6,lote,canal,cartao_presencial,5815,55,5815,1.0,0.836364,0.407080,0.809571,3320.91,3320.91,571.093723,NaN,False,comparado,13.0,7,0.09
7,lote,canal,pix,8609,107,8609,1.0,0.831776,0.494444,0.837898,2866.59,2866.59,332.975955,NaN,False,comparado,13.0,7,0.09


In [8]:
alertas_saida_14 = relatorio_14["saida"].loc[
    relatorio_14["saida"]["semana"].eq(14) & relatorio_14["saida"]["alerta"]
]
assert len(alertas_saida_14) == 1
print("Alerta observado na semana 14")
display(alertas_saida_14[["semana", "bloqueios", "taxa_bloqueio", "delta_taxa_bloqueio_pp"]])
mostrar_decisao(14)

Alerta observado na semana 14


,semana,bloqueios,taxa_bloqueio,delta_taxa_bloqueio_pp
2,14.0,812,0.039643,1.191901


,semana,versao_modelo,versao_contrato,retreinou,acao
1,14,7,1.0,False,"Manter a v7 e o corte de 0,09. Registrar a hip..."


*Semana 14 — o que eu vi e o que vou fazer:*

O lote 14 teve 812 retenções entre 20.483 transações (3,96%, ante 2,65% na semana 13), acima do limite frente à validação. As contestações da semana 13 fecharam o custo em R$ 10.986,30, com recall de 84,86% e 270 clientes com transações legítimas retidas. A investigação encontrou um provável deslocamento de três horas nos dados da Vértice. A decisão registrada foi manter a v7 com corte 0,09 e levar a evidência à Plataforma para confirmar o fuso e pedir a reemissão do lote.

## Semana 15

**Chegam:** `dados/parte2/transacoes_semana_15.csv` e `dados/parte2/rotulos_semana_14.csv`.

1. Envie o lote na tela **Lotes** do sistema e, **depois**, os rótulos na tela **Rótulos**.
   Os dados já pontuados da semana saem na própria tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 15 contra a referência (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - **performance da semana 14**, que acabou de ganhar rótulo: recall, precision, PR-AUC e custo, globais **e por segmento**; compare com a referência e com as semanas anteriores;
3. Leia o relatório e escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 15 na tela **Decisões** do sistema.

Se você decidir agir agora, faça a investigação e a ação (seções I, D e A, no fim do notebook) **antes** de seguir para a próxima semana. Registre a decisão desta semana com o que você fez.

> **Antes da semana 16:** leia o **Anexo 2** da demanda.

In [9]:
relatorio_15 = mostrar_relatorio(15)

continuas


,variavel,n_referencia,n_lote,invalidos_referencia,invalidos_lote,status,media_referencia,media_lote,D,p_valor,alerta
0,valor_transacao,39497,20062,0,0,calculado,185.699292,185.123384,0.010294,1.184433e-01,False
1,qtd_transacoes_1h,39497,20062,0,0,calculado,0.365445,0.357542,0.004307,9.650764e-01,False
2,valor_medio_30d,39497,20062,0,0,calculado,118.092749,118.510160,0.006067,7.092911e-01,False
3,razao_valor_media,39497,20062,0,0,calculado,1.581537,1.563000,0.006664,5.935493e-01,False
4,distancia_ultima_transacao_km,39497,20062,0,0,calculado,22.099109,23.156585,0.005466,8.194489e-01,False
5,idade_conta_dias,39497,20062,0,0,calculado,816.248196,836.441731,0.058070,1.942368e-39,False
6,score,39497,20062,0,0,calculado,0.014888,0.017152,0.075614,1.339415e-66,False


discretas


,variavel,n_referencia,n_lote,categorias,categoria_maior_mudanca,maior_mudanca_pp,p_valor,alerta
0,hora_transacao,39497,20062,24,23,4.466390,0.000000,False
1,dia_semana,39497,20062,7,3,0.425010,0.441401,False
2,canal,39497,20062,3,pix,0.135920,0.926851,False
3,dispositivo_novo,39497,20062,2,1,0.250262,0.129486,False
4,destinatario_novo,39497,20062,2,0,0.652616,0.073380,False
5,categoria_estabelecimento,39497,20062,10,eletronicos,0.261562,0.973066,False
6,uf_cliente,39497,20062,27,RS,0.393315,0.788700,False


saida


,conjunto,transacoes,bloqueios,taxa_bloqueio,valor_bloqueado,valor_total,pct_valor_bloqueado,delta_taxa_bloqueio_pp,delta_pct_valor_bloqueado_pp,alerta,semana,versao_modelo,threshold
0,referencia,39497,1095,0.027724,603268.81,7334564.93,0.082250,0.000000,0.000000,False,NaN,7,0.09
1,lote,19419,515,0.026520,300237.57,3579130.56,0.083886,-0.120321,0.163550,False,13.0,7,0.09
2,lote,20483,812,0.039643,391360.35,3793428.23,0.103168,1.191901,2.091787,True,14.0,7,0.09
3,lote,20062,750,0.037384,349188.86,3713945.32,0.094021,0.966048,1.177089,False,15.0,7,0.09


performance


,conjunto,segmento,valor_segmento,transacoes,fraudes,rotuladas,cobertura_rotulos,recall,precision,pr_auc,custo_observado,custo,custo_por_mil,motivos_alerta,alerta,status_comparacao,semana,versao_modelo,threshold
0,referencia,global,todos,39497,600,39497,1.0,0.851667,0.466667,0.818789,18866.83,18866.83,477.677545,NaN,False,referencia,NaN,7,0.09
1,referencia,canal,cartao_online,9871,201,9871,1.0,0.905473,0.434368,0.860174,5858.16,5858.16,593.471786,NaN,False,referencia,NaN,7,0.09
2,referencia,canal,cartao_presencial,11828,113,11828,1.0,0.761062,0.352459,0.630069,5731.43,5731.43,484.564592,NaN,False,referencia,NaN,7,0.09
3,referencia,canal,pix,17798,286,17798,1.0,0.849650,0.562500,0.849862,7277.24,7277.24,408.879649,NaN,False,referencia,NaN,7,0.09
4,lote,global,todos,19419,284,19419,1.0,0.848592,0.467961,0.838318,10986.30,10986.30,565.750039,NaN,False,comparado,13.0,7,0.09
5,lote,canal,cartao_online,4995,122,4995,1.0,0.868852,0.477477,0.854422,4798.80,4798.80,960.720721,NaN,False,comparado,13.0,7,0.09
6,lote,canal,cartao_presencial,5815,55,5815,1.0,0.836364,0.407080,0.809571,3320.91,3320.91,571.093723,NaN,False,comparado,13.0,7,0.09
7,lote,canal,pix,8609,107,8609,1.0,0.831776,0.494444,0.837898,2866.59,2866.59,332.975955,NaN,False,comparado,13.0,7,0.09
8,lote,global,todos,20483,303,20483,1.0,0.841584,0.314039,0.753631,15065.62,15065.62,735.518235,"queda_precision, queda_pr_auc, aumento_custo_p...",True,comparado,14.0,7,0.09
9,lote,canal,cartao_online,5149,107,5149,1.0,0.841121,0.276923,0.784469,6175.85,6175.85,1199.427073,queda_precision,True,comparado,14.0,7,0.09


In [10]:
comparacao_15 = pd.read_csv(pasta_relatorios / "semana_15" / "saida_sem15.csv")
assert comparacao_15["retidas"].tolist() == [750, 506]
print("Semana 15 antes e depois da correção")
display(comparacao_15)
mostrar_decisao(15)

Semana 15 antes e depois da correção


,cenario,retidas,taxa_retencao,valor_retido,clientes_retidos
0,original,750,0.037384,349188.86,741
1,corrigida,506,0.025222,270299.69,501


,semana,versao_modelo,versao_contrato,retreinou,acao
2,15,7,1.0,False,Aplicar a correção v1.1.1 ao lote da semana 15...


*Semana 15 — o que eu vi e o que vou fazer:*

O sistema recebeu 20.062 transações na semana 15 e reteve 750 (3,74%). O perfil horário continuou deslocado. Com as 20.483 contestações da semana 14, o custo observado daquela semana foi de R$ 15.065,62, com 553 clientes que tiveram transações legítimas retidas. O teste com a entrada ajustada para o horário de Brasília reduziu o custo simulado a R$ 10.525,93 e recuperou o desempenho com a mesma v7.

**Decisão registrada.** Apliquei a correção de horário v1.1.1 ao lote 15 e repontuei as 20.062 transações com a v7 e corte de 0,09. O sistema passou de 750 para 506 retenções (3,74% → 2,52%) e de 741 para 501 clientes com retenção. As 20.062 predições originais foram preservadas no banco. O lote da semana 14 manteve as 812 retenções da operação original. O custo e o recall da semana 15 serão apurados após a chegada das contestações. A referência congelada e o contrato v1.0 foram mantidos. A auditoria está no MLflow, run `8b3dff66abf04f80896908490dc00f8d`.

## Semana 16

**Chegam:** `dados/parte2/transacoes_semana_16.csv` e `dados/parte2/rotulos_semana_15.csv`.

1. Envie o lote na tela **Lotes** do sistema e, **depois**, os rótulos na tela **Rótulos**.
   Os dados já pontuados da semana saem na própria tela **Lotes**, no link *baixar dados*.
2. Rode o seu relatório da semana:
   - drift do lote 16 contra a referência (features e score);
   - saída do modelo (taxa de bloqueio e valor bloqueado), comparada com a referência e com as semanas anteriores;
   - **performance da semana 15**, que acabou de ganhar rótulo: recall, precision, PR-AUC e custo, globais **e por segmento**; compare com a referência e com as semanas anteriores;
3. Leia o relatório e escreva abaixo, em 2 a 4 linhas: **o que você viu e o que vai fazer**.
4. Registre a decisão da semana 16 na tela **Decisões** do sistema.

Se você decidir agir agora, faça a investigação e a ação (seções I, D e A, no fim do notebook) **antes** de seguir para a próxima semana. Registre a decisão desta semana com o que você fez.

> As contestações da semana 16 só chegam na parte 3. O que você já sabe sobre ela vem das features e do score.

In [11]:
relatorio_16 = mostrar_relatorio(16)

continuas


,variavel,n_referencia,n_lote,invalidos_referencia,invalidos_lote,status,media_referencia,media_lote,D,p_valor,alerta
0,valor_transacao,39497,19959,0,0,calculado,185.699292,182.135111,0.005073,8.827719e-01,False
1,qtd_transacoes_1h,39497,19959,0,0,calculado,0.365445,0.371111,0.003339,9.983617e-01,False
2,valor_medio_30d,39497,19959,0,0,calculado,118.092749,116.855270,0.008128,3.432628e-01,False
3,razao_valor_media,39497,19959,0,0,calculado,1.581537,1.577195,0.008352,3.115421e-01,False
4,distancia_ultima_transacao_km,39497,19959,0,0,calculado,22.099109,23.207691,0.006088,7.071084e-01,False
5,idade_conta_dias,39497,19959,0,0,calculado,816.248196,849.804750,0.062428,2.299089e-45,False
6,score,39497,19959,0,0,calculado,0.014888,0.019604,0.084878,1.499270e-83,False


discretas


,variavel,n_referencia,n_lote,categorias,categoria_maior_mudanca,maior_mudanca_pp,p_valor,alerta
0,hora_transacao,39497,19959,24,23,4.628682,0.000000,False
1,dia_semana,39497,19959,7,4,0.679785,0.071476,False
2,canal,39497,19959,3,cartao_presencial,0.160141,0.914005,False
3,dispositivo_novo,39497,19959,2,1,0.142096,0.397966,False
4,destinatario_novo,39497,19959,2,1,0.468659,0.202962,False
5,categoria_estabelecimento,39497,19959,10,servicos,0.397609,0.951421,False
6,uf_cliente,39497,19959,27,RS,0.482822,0.485466,False


saida


,conjunto,transacoes,bloqueios,taxa_bloqueio,valor_bloqueado,valor_total,pct_valor_bloqueado,delta_taxa_bloqueio_pp,delta_pct_valor_bloqueado_pp,alerta,semana,versao_modelo,threshold
0,referencia,39497,1095,0.027724,603268.81,7334564.93,0.082250,0.000000,0.000000,False,NaN,7,0.09
1,lote,19419,515,0.026520,300237.57,3579130.56,0.083886,-0.120321,0.163550,False,13.0,7,0.09
2,lote,20483,812,0.039643,391360.35,3793428.23,0.103168,1.191901,2.091787,True,14.0,7,0.09
3,lote,20062,506,0.025222,270299.69,3713945.32,0.072780,-0.250181,-0.947045,False,15.0,7,0.09
4,lote,19959,803,0.040232,351309.66,3635234.69,0.096640,1.250885,1.439004,True,16.0,7,0.09


performance


,conjunto,segmento,valor_segmento,transacoes,fraudes,rotuladas,cobertura_rotulos,recall,precision,pr_auc,custo_observado,custo,custo_por_mil,motivos_alerta,alerta,status_comparacao,semana,versao_modelo,threshold
0,referencia,global,todos,39497,600,39497,1.0,0.851667,0.466667,0.818789,18866.83,18866.83,477.677545,NaN,False,referencia,NaN,7,0.09
1,referencia,canal,cartao_online,9871,201,9871,1.0,0.905473,0.434368,0.860174,5858.16,5858.16,593.471786,NaN,False,referencia,NaN,7,0.09
2,referencia,canal,cartao_presencial,11828,113,11828,1.0,0.761062,0.352459,0.630069,5731.43,5731.43,484.564592,NaN,False,referencia,NaN,7,0.09
3,referencia,canal,pix,17798,286,17798,1.0,0.849650,0.562500,0.849862,7277.24,7277.24,408.879649,NaN,False,referencia,NaN,7,0.09
4,lote,global,todos,19419,284,19419,1.0,0.848592,0.467961,0.838318,10986.30,10986.30,565.750039,NaN,False,comparado,13.0,7,0.09
5,lote,canal,cartao_online,4995,122,4995,1.0,0.868852,0.477477,0.854422,4798.80,4798.80,960.720721,NaN,False,comparado,13.0,7,0.09
6,lote,canal,cartao_presencial,5815,55,5815,1.0,0.836364,0.407080,0.809571,3320.91,3320.91,571.093723,NaN,False,comparado,13.0,7,0.09
7,lote,canal,pix,8609,107,8609,1.0,0.831776,0.494444,0.837898,2866.59,2866.59,332.975955,NaN,False,comparado,13.0,7,0.09
8,lote,global,todos,20483,303,20483,1.0,0.841584,0.314039,0.753631,15065.62,15065.62,735.518235,"queda_precision, queda_pr_auc, aumento_custo_p...",True,comparado,14.0,7,0.09
9,lote,canal,cartao_online,5149,107,5149,1.0,0.841121,0.276923,0.784469,6175.85,6175.85,1199.427073,queda_precision,True,comparado,14.0,7,0.09


In [12]:
pasta_semana_16 = pasta_relatorios / "semana_16"
comparacao_16 = pd.read_csv(pasta_semana_16 / "comparacao_horario.csv").query("semana == 16")
assert comparacao_16["retidas"].tolist() == [803, 546]
print("Semana 16 antes e depois da correção")
display(comparacao_16[["cenario", "transacoes", "retidas", "taxa_retencao", "valor_retido", "clientes_retidos"]])
mostrar_decisao(16)

Semana 16 antes e depois da correção

,cenario,transacoes,retidas,taxa_retencao,valor_retido,clientes_retidos
5,original,19959,803,0.040232,351309.66,789
6,corrigida,19959,546,0.027356,273064.10,538


,semana,versao_modelo,versao_contrato,retreinou,acao
3,16,7,1.0,False,Aplicar novamente a correção de horário v1.1.1...


*Semana 16 — o que eu vi e o que vou fazer:*

O lote 16 repetiu o desvio: 266 registros apareciam fora da janela e voltaram à semana correta após a conversão. As retenções passaram de 803 (4,02%) para 546 (2,74%), próximas dos 2,77% da referência. As contestações da 15 confirmaram R$ 9.759,94 de custo após a correção e 303 clientes com transações legítimas retidas. A decisão da 16 foi reaplicar a correção, manter a v7 e revisar os segmentos apontados pela Central quando houver rótulos da semana 16.

## I. Investigação

Use esta seção quando algo chamar a sua atenção, seja um alerta do monitoramento, um número do painel ou um e-mail. Responda com evidência, ou seja, uma tabela ou um gráfico para cada afirmação:

1. **O que mudou?** Quais colunas, e como: a distribuição inteira, a cauda, ou um grupo específico?
2. **Desde quando?** Em que semana começou? A mudança é abrupta ou gradual?
3. **Onde?** Em todas as transações ou em um segmento (canal, horário, tipo de cliente...)?
4. **Que tipo de problema é?** Separe três possibilidades:
   - o **dado** mudou de formato ou de significado, e o mundo continua igual;
   - o **mundo** mudou: os clientes ou os golpistas passaram a se comportar diferente;
   - a **relação** entre as features e a fraude mudou, e o modelo ficou errado.

   Quais hipóteses você testou, e como descartou as outras?
5. **Quanto custa?** Mostre o impacto em custo, bloqueio e recall por semana, global e no segmento afetado, e compare com o período de referência.

> Lembre do relatório zero: com 20 mil linhas, quase todo p-valor é minúsculo. Olhe o tamanho do efeito.

In [13]:
# Revisão feita na semana 15, após a chegada das contestações da semana 14.
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import Pipeline

import config
from app.contrato import carregar, nomes_features, validar
from avaliar_modelos import carregar_historico, medir
from features import criar_preprocessador
from monitoramento import custo_decisao, drift_discretas

contrato_v1 = carregar()
colunas_v7 = nomes_features(contrato_v1)
referencia_v7 = pd.read_csv("referencia/validacao_11_12.csv", dtype={"id_transacao": str})
pasta_dados = Path("dados/parte2")
corte_v7 = 0.09

def ler_lote_operacao(semana):
    bruto = pd.read_csv(pasta_dados / f"transacoes_semana_{semana}.csv",
                        dtype={"id_transacao": str, "id_cliente": str})
    lote, erros = validar(bruto, contrato_v1)
    if erros:
        raise ValueError(erros)
    return lote

def juntar_contestacoes(lote, semana):
    rotulos = pd.read_csv(pasta_dados / f"rotulos_semana_{semana}.csv",
                          usecols=["id_transacao", "fraude"], dtype={"id_transacao": str})
    dados = lote.merge(rotulos, on="id_transacao", how="left", validate="one_to_one")
    if len(dados) != len(rotulos) or not dados["fraude"].isin([0, 1]).all():
        raise ValueError(f"Rótulos incompletos ou inconsistentes na semana {semana}")
    return dados

sem13 = juntar_contestacoes(ler_lote_operacao(13), 13)
sem14_bruta = juntar_contestacoes(ler_lote_operacao(14), 14)
sem15_bruta = ler_lote_operacao(15)

def simular_fuso_vertice(lote):
    horario = pd.to_datetime(lote["data_transacao"], errors="raise") - pd.Timedelta(hours=3)
    return lote.assign(data_transacao=horario, hora_transacao=horario.dt.hour,
                       dia_semana=horario.dt.dayofweek)

sem14_corrigida = simular_fuso_vertice(sem14_bruta)
sem15_corrigida = simular_fuso_vertice(sem15_bruta)

In [14]:
def proporcao_por_hora(lote):
    return lote["hora_transacao"].value_counts(normalize=True).reindex(range(24), fill_value=0)

def distancia_horaria(base, lote):
    return float((proporcao_por_hora(base) - proporcao_por_hora(lote)).abs().sum() * 50)

def fora_da_semana(lote, semana):
    inicio = pd.Timestamp("2026-08-24") + pd.Timedelta(weeks=semana - 13)
    horario = pd.to_datetime(lote["data_transacao"])
    return int((horario.lt(inicio) | horario.ge(inicio + pd.Timedelta(weeks=1))).sum())

verificacao = pd.DataFrame([
    {"semana": semana, "cenario": nome, "datas_fora_da_semana": fora_da_semana(lote, semana),
     "distancia_horaria_da_13_pp": distancia_horaria(sem13, lote),
     "maior_diferenca_por_hora_vs_11_12_pp": float(drift_discretas(
         referencia_v7, lote, ["hora_transacao"]).loc[0, "maior_mudanca_pp"])}
    for semana, nome, lote in ((13, "original", sem13), (14, "original", sem14_bruta),
                               (14, "ajuste -3h", sem14_corrigida), (15, "original", sem15_bruta),
                               (15, "ajuste -3h", sem15_corrigida))
])
display(verificacao)

,semana,cenario,datas_fora_da_semana,distancia_horaria_da_13_pp,maior_diferenca_por_hora_vs_11_12_pp
0,13,original,0,0.000000,0.457379
1,14,original,283,22.806115,4.563415
2,14,ajuste -3h,0,1.568973,0.308866
3,15,original,257,21.892753,4.466390
4,15,ajuste -3h,0,1.615524,0.291784


In [15]:
exportacao_v7 = Path("entrega/modelo_exportado")
if exportacao_v7.is_dir():
    modelo_v7 = mlflow.sklearn.load_model(str(exportacao_v7))
else:
    mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
    modelo_v7 = mlflow.sklearn.load_model(f"models:/{config.NOME_MODELO}/7")

scores_v7 = {
    nome: modelo_v7.predict_proba(lote[colunas_v7])[:, 1]
    for nome, lote in (("13", sem13), ("14_original", sem14_bruta), ("14_corrigida", sem14_corrigida),
                       ("15_original", sem15_bruta), ("15_corrigida", sem15_corrigida))
}

# O corte alternativo é escolhido na 13 e avaliado na 14, que fica fora da busca.
def melhor_corte(lote, scores):
    return min((i / 100 for i in range(1, 100)), key=lambda c: custo_decisao(
        lote["fraude"], lote["valor_transacao"], scores, c, config.CUSTO_FALSO_POSITIVO))

corte_alternativo = melhor_corte(sem13, scores_v7["13"])
def resultado(nome, semana, lote, scores, corte):
    return {"cenario": nome, "semana": semana, "corte": corte, "transacoes": len(lote),
            **medir(scores >= corte, lote, scores)}

comparacao_v7 = pd.DataFrame([
    resultado(f"v7 {dado} | {tipo}", 14, lote, scores_v7[f"14_{dado}"], corte)
    for dado, lote in (("original", sem14_bruta), ("corrigida", sem14_corrigida))
    for tipo, corte in (("corte vigente", corte_v7), ("corte escolhido na 13", corte_alternativo))
])
comparacao_canais = pd.DataFrame([
    {"canal": canal, **resultado(dado, 14, lote.loc[mask], scores_v7[f"14_{dado}"][mask], corte_v7)}
    for canal in sorted(sem14_bruta["canal"].unique())
    for dado, lote in (("original", sem14_bruta), ("corrigida", sem14_corrigida))
    for mask in [lote["canal"].eq(canal).to_numpy()]
])
saida_sem15 = pd.DataFrame([
    {"cenario": dado, "retidas": int(retidas.sum()), "taxa_retencao": float(retidas.mean()),
     "valor_retido": float(lote.loc[retidas, "valor_transacao"].sum()),
     "clientes_retidos": lote.loc[retidas, "id_cliente"].nunique()}
    for dado, lote in (("original", sem15_bruta), ("corrigida", sem15_corrigida))
    for retidas in [scores_v7[f"15_{dado}"] >= corte_v7]
])
display(comparacao_v7)
display(comparacao_canais)
display(saida_sem15)

,cenario,semana,corte,transacoes,custo,valor_fraudes_aprovadas,fraudes_aprovadas,legitimas_bloqueadas,clientes_legitimos_afetados,taxa_bloqueio,recall,precision,pr_auc
0,v7 original | corte vigente,14,0.09,20483,15065.62,6710.62,48,557,553,0.039643,0.841584,0.314039,0.753631
1,v7 original | corte escolhido na 13,14,0.17,20483,13493.51,10568.51,67,195,194,0.021042,0.778878,0.547564,0.753631
2,v7 corrigida | corte vigente,14,0.09,20483,10525.93,6130.93,43,293,291,0.026998,0.858086,0.470163,0.815881
3,v7 corrigida | corte escolhido na 13,14,0.17,20483,11662.23,9862.23,61,120,120,0.017673,0.798680,0.668508,0.815881


,canal,cenario,semana,corte,transacoes,custo,valor_fraudes_aprovadas,fraudes_aprovadas,legitimas_bloqueadas,clientes_legitimos_afetados,taxa_bloqueio,recall,precision,pr_auc
0,cartao_online,original,14,0.09,5149,6175.85,2650.85,17,235,234,0.063119,0.841121,0.276923,0.784469
1,cartao_online,corrigida,14,0.09,5149,4336.34,2386.34,15,130,129,0.043115,0.859813,0.414414,0.824801
2,cartao_presencial,original,14,0.09,6003,3077.49,1352.49,12,115,115,0.028486,0.823529,0.327485,0.724261
3,cartao_presencial,corrigida,14,0.09,6003,2372.49,1352.49,12,68,68,0.020656,0.823529,0.451613,0.776190
4,pix,original,14,0.09,9331,5812.28,2707.28,19,207,206,0.033866,0.851562,0.344937,0.760720
5,pix,corrigida,14,0.09,9331,3817.10,2392.10,16,95,95,0.022184,0.875000,0.541063,0.831806


,cenario,retidas,taxa_retencao,valor_retido,clientes_retidos
0,original,750,0.037384,349188.86,741
1,corrigida,506,0.025222,270299.69,501


In [16]:
# O mesmo lote bruto pode parecer normal se a referência também contiver o erro.
comparacao_referencia = pd.DataFrame([
    {"referencia": nome, "distancia_horaria_da_semana_15_bruta_pp": distancia_horaria(base, sem15_bruta),
     "distancia_horaria_da_semana_15_corrigida_pp": distancia_horaria(base, sem15_corrigida)}
    for nome, base in (("11–12 congelada", referencia_v7), ("13 saudável", sem13),
                       ("14 original", sem14_bruta), ("14 corrigida", sem14_corrigida))
])
display(comparacao_referencia)

,referencia,distancia_horaria_da_semana_15_bruta_pp,distancia_horaria_da_semana_15_corrigida_pp
0,11–12 congelada,21.492204,1.100170
1,13 saudável,21.892753,1.615524
2,14 original,1.322042,22.257196
3,14 corrigida,21.529640,1.322042


*Diagnóstico — o que aconteceu, desde quando, onde, de que tipo, quanto custa:*

### Semana 14 — horário dos registros após a migração

O contrato define `hora_transacao` no horário de Brasília. Na semana 13, a hora e o dia coincidem com `data_transacao`. Na semana 14, também coincidem, mas 283 registros aparecem entre 00h00 e 02h59 da segunda-feira seguinte. Ao subtrair três horas de `data_transacao`, todos voltam à semana 14 e o dia da semana muda em 2.167 registros. Entre os deslocamentos inteiros de −12 a +11 horas, −3 foi o que melhor aproximou a distribuição de horários da semana 13 em todos os canais.

| Canal | Distância do perfil horário da semana 13 | Após simular −3h | Retenções reais | Retenções simuladas |
|---|---:|---:|---:|---:|
| Global | 22,81 pp | 1,57 pp | 812 | 553 |
| Cartão online | 22,20 pp | 2,80 pp | 325 | 222 |
| Cartão presencial | 23,22 pp | 3,78 pp | 171 | 124 |
| Pix | 22,92 pp | 2,58 pp | 316 | 207 |

A distância é metade da soma das diferenças absolutas entre as proporções das 24 horas. O teste usou a v7 com o corte de 0,09 e recalculou a hora e o dia a partir do horário ajustado. A taxa simulada foi de 2,70%. O valor retido passaria de R$ 391.360,35 a R$ 276.591,76, e o número de clientes com retenção, de 800 a 547. A classificação dessas retenções e seu custo dependem das contestações da semana 14.

Com todos os rótulos da semana 13, o custo foi de R$ 6.876,30 em 43 fraudes aprovadas mais R$ 4.110,00 pelas 274 transações legítimas retidas. Foram afetados 270 clientes distintos com transações legítimas. A precisão foi de 46,80% e a PR-AUC de 0,8383. Esses resultados correspondem à semana anterior à migração.

### Revisão da semana 14 com os rótulos disponíveis na semana 15

As aulas 2 a 5 do outro projeto orientaram os testes: comparar a população e o significado dos campos, testar uma correção reversível, validar candidatos fora do treino e verificar se a relação com o rótulo volta ao padrão depois do ajuste. Os rótulos usados aqui vão até a semana 14.

Na semana 15, 257 registros vieram com horário fora da janela do lote. O ajuste de três horas recolocou todos na janela e reduziu a distância do perfil da semana 13 de 21,89 para 1,62 pp. Os resultados se repetem nas duas semanas após a migração. A incidência de fraude foi de 1,46% na semana 13, 1,48% na 14 e 1,52% na validação. Depois do ajuste, a PR-AUC da 14 voltou de 0,754 para 0,816, próxima dos 0,819 da validação. O conjunto aponta para um problema de horário na entrada.

| Semana 14 | Custo | Fraudes detectadas | Transações legítimas retidas | Clientes legítimos afetados |
|---|---:|---:|---:|---:|
| Pontuação registrada | R$ 15.065,62 | 84,16% | 557 | 553 |
| Simulação com horário ajustado | R$ 10.525,93 | 85,81% | 293 | 291 |

No cenário corrigido, as fraudes aprovadas caem de 48 para 43 e o valor correspondente, de R$ 6.710,62 para R$ 6.130,93. A economia simulada é de R$ 4.539,69. O lote da semana 15, ainda sem rótulos, passaria de 750 para 506 retenções no mesmo teste.

### Semana 16 — investigação do relato da Central

As compras entre 18h e 22h foram comparadas pelo horário de Brasília nos dois cenários, usando as mesmas transações e o mesmo corte. Na semana 15, a correção reduziu as retenções de transações legítimas nessa faixa de 303 para 66. Em restaurantes nesse horário, passaram de 8 para 0 e, em supermercados, de 21 para 8. Para Pix a destinatários conhecidos, foram de 102 para 34. Os recortes podem se sobrepor. A análise mede transações e clientes. O acompanhamento das ligações cabe à Central.

| Comparação da semana 15 | Entrada original | Entrada corrigida |
|---|---:|---:|
| Custo | R$ 14.503,03 | R$ 9.759,94 |
| Recall | 78,14% | 81,38% |
| Precisão | 25,73% | 39,72% |
| Clientes com transações legítimas retidas | 553 | 303 |

A precisão da semana 15 ficou abaixo dos 46,67% da referência e acionou o alerta. A incidência de fraude também caiu de 1,52% na referência para 1,23% na semana 15. O custo corrigido ficou em R$ 486,49 por mil transações, frente a R$ 477,68 na referência. O desempenho merece acompanhamento com os próximos rótulos. A semana 16 corrigida teve 546 retenções, das quais 111 ocorreram entre 18h e 22h. O custo, o recall e os bloqueios indevidos da 16 serão medidos quando chegarem as contestações.

### Hipóteses avaliadas

| Hipótese | Evidência e conclusão |
|---|---|
| Mudança no dado recebido | O desvio começou na semana 14, após a migração, e se repetiu nos três canais até a semana 16. A conversão para Brasília recolocou 283, 257 e 266 registros dentro das respectivas semanas. No lote 16, a maior diferença de proporção por hora frente à referência caiu de 4,63 para 0,27 ponto percentual. |
| Mudança no comportamento dos clientes | No lote 16, as maiores diferenças de proporção por canal e categoria frente à referência foram de 0,16 e 0,40 ponto percentual. O mesmo ajuste de horário recuperou o perfil dos três canais. A evidência favorece o deslocamento no arquivo como explicação para o aumento de retenções. |
| Mudança na relação entre transações e fraude | Na semana 14, com a mesma v7, o ajuste elevou a PR-AUC de 0,754 para 0,816, perto dos 0,819 da validação. Na semana 15 a precisão ainda ficou em 39,72%, frente a 46,67% na referência. O custo por mil voltou a ficar próximo da validação. A relação com fraude segue em acompanhamento quando chegarem os rótulos da 16. |

## D. Decisão

Para cada alternativa abaixo, diga em uma linha se você a considerou e por que a escolheu ou a descartou. Mais de uma pode valer ao mesmo tempo.

| Alternativa | Considerei? Escolhi? Por quê? |
|---|---|
| Não fazer nada | **Considerei. Escolhi manter a operação na semana 13**, com 2,65% de retenção e sem alertas. Nas semanas seguintes, o desvio persistiu. Na 14, o custo observado chegou a R$ 15.065,62 e 553 clientes tiveram transações legítimas retidas. |
| Mudar o threshold | **Considerei. Descartei.** O corte 0,17, escolhido na semana 13 e testado na 14, baixou o recall do lote bruto de 84,16% para 77,89%. No lote corrigido, 0,17 custou R$ 11.662,23, diante de R$ 10.525,93 com 0,09. |
| Corrigir o dado ou o pré-processamento, sem retreinar | **Considerei. Escolhi corrigir a entrada.** A simulação da semana 14 reduziu o custo em R$ 4.539,69 e elevou o recall de 84,16% para 85,81%. Apliquei a conversão para Brasília nos lotes 15 e 16 e repontuei com a mesma v7. |
| Atualizar a referência do monitoramento | **Considerei. Descartei.** Usar a semana 14 bruta como referência reduziria de 21,49 para 1,32 pp a distância do perfil horário da semana 15 bruta, pois ambos os lotes carregavam o desvio. Mantive as semanas 11 e 12. |
| Mudar o contrato de features | **Considerei. Descartei.** O contrato v1.0 já exige hora de Brasília. A correção restaurou essa regra e a checagem de janela confirmou as datas de cada lote. |
| Retreinar o modelo | **Considerei. Descartei.** Com a semana 13 no treino, a economia adicional na semana 14 corrigida foi de R$ 142,82, com o mesmo recall. Na validação congelada, o custo subiu de R$ 18.866,83 para R$ 18.942,12. Retirar hora e dia elevou o custo da semana 14 corrigida a R$ 10.682,44 e reduziu o recall a 84,16%. |

Se a decisão for **retreinar**, diga com quais semanas e por quê. Mostre também, em dados que o novo modelo não viu, que ele é melhor que o atual.

Se a decisão for **não retreinar**, mostre a evidência de que o modelo atual continua adequado.

*Sua decisão e a justificativa:*

**Semana 13.** Mantive a v7 e o corte de 0,09. A saída estava próxima da referência e as contestações chegariam na rodada seguinte.

**Semana 14.** Mantive a pontuação registrada e investiguei o desvio de horário após o alerta. Os rótulos disponíveis nessa rodada eram da semana 13.

**Semanas 15 e 16.** Corrigi o fuso na entrada e repontuei com a v7, mantendo o contrato v1.0 e a referência das semanas 11 e 12. A escolha preserva o desempenho histórico. A correção recuperou a taxa de retenção e o custo por mil da semana 15 ficou em R$ 486,49, próximo dos R$ 477,68 da validação. A queda de precisão da semana 15 permanece em acompanhamento.

**Validação dos retreinos de controle.** Os dois candidatos usaram as semanas 1 a 10 e 13 no treino, com corte fixo de 0,09. As semanas 11, 12 e 14 ficaram fora do ajuste. A comparação abaixo sustenta a escolha pela v7 com a entrada corrigida.

In [17]:
# Os resultados dos dois retreinos executados estão no relatório arquivado.
REFAZER_RETREINOS = False
if REFAZER_RETREINOS:
    # Dois retreinos de controle. A semana 14 não participa do ajuste.
    historico = carregar_historico(contrato_v1)
    treino_novo = pd.concat([historico.loc[historico["semana"].between(1, 10)], sem13], ignore_index=True)
    controle_congelado = historico.loc[historico["semana"].isin([11, 12])]
    estimador_v7 = modelo_v7.steps[-1][1]

    resultados_retreino = []
    for nome, excluir in (("Retreino com semana 13", set()),
                          ("Retreino sem hora e dia", {"hora_transacao", "dia_semana"})):
        contrato_teste = {**contrato_v1, "features": [
            f for f in contrato_v1["features"] if f["nome"] not in excluir]}
        colunas = nomes_features(contrato_teste)
        modelo_teste = Pipeline([("preprocessador", criar_preprocessador(contrato_teste)),
                                 ("floresta", clone(estimador_v7))])
        modelo_teste.fit(treino_novo[colunas], treino_novo["fraude"])
        for semana, lote in (("11–12", controle_congelado), (14, sem14_corrigida)):
            scores = modelo_teste.predict_proba(lote[colunas])[:, 1]
            resultados_retreino.append(resultado(nome, semana, lote, scores, corte_v7))

    for semana, lote in (("11–12", controle_congelado), (14, sem14_corrigida)):
        scores = modelo_v7.predict_proba(lote[colunas_v7])[:, 1]
        resultados_retreino.append(resultado("v7 com entrada no horário local", semana, lote, scores, corte_v7))

    comparacao_retreino = pd.DataFrame(resultados_retreino)
else:
    comparacao_retreino = pd.read_csv(pasta_relatorios / "semana_15" / "comparacao_retreino.csv")
display(comparacao_retreino[["cenario", "semana", "corte", "custo", "recall", "precision", "pr_auc"]])

,cenario,semana,corte,custo,recall,precision,pr_auc
0,Retreino com semana 13,11–12,0.09,18942.12,0.841667,0.484180,0.820739
1,Retreino com semana 13,14,0.09,10383.11,0.858086,0.475320,0.818489
2,Retreino sem hora e dia,11–12,0.09,19769.87,0.830000,0.490157,0.797581
3,Retreino sem hora e dia,14,0.09,10682.44,0.841584,0.477528,0.799914
4,v7 com entrada no horário local,11–12,0.09,18866.83,0.851667,0.466667,0.818789
5,v7 com entrada no horário local,14,0.09,10525.93,0.858086,0.470163,0.815881


In [18]:
alternativas_corte = comparacao_v7.loc[:, ["cenario", "corte", "custo", "recall", "clientes_legitimos_afetados"]]
escolhido = alternativas_corte.loc[alternativas_corte["cenario"].eq("v7 corrigida | corte vigente")].iloc[0]
assert escolhido["corte"] == 0.09 and escolhido["custo"] == 10525.93
print("Comparação dos cortes na semana 14, fora da escolha do corte alternativo")
display(alternativas_corte)

Comparação dos cortes na semana 14, fora da escolha do corte alternativo


,cenario,corte,custo,recall,clientes_legitimos_afetados
0,v7 original | corte vigente,0.09,15065.62,0.841584,553
1,v7 original | corte escolhido na 13,0.17,13493.51,0.778878,194
2,v7 corrigida | corte vigente,0.09,10525.93,0.858086,291
3,v7 corrigida | corte escolhido na 13,0.17,11662.23,0.798680,120


## A. Ação e rastreabilidade

Faça o que você decidiu e deixe tudo rastreável.

**Se mudou o modelo ou o pipeline**
- Registre uma **nova versão** do `antifraude` com as tags `threshold` e `motivo` e uma descrição. Depois, mova o `@champion`.
- Se o pipeline usa código seu (`features.py`), registre com `code_paths=["features.py"]`.
- Refaça o teste da parte 1: os scores do `POST /predict` precisam ser idênticos aos do modelo carregado do MLflow.
- Decida quais semanas **repontuar** (botão *Repontuar* na tela **Lotes**) e justifique. Repontuar uma semana que não precisava também é um erro.

**Se mudou o contrato**
- Suba a `versao` no `contrato/contrato_features.yaml` e registre a mudança no `changelog` do próprio arquivo.
- O sistema relê o contrato a cada lote. Confira na tela **Modelo** que a versão nova está em vigor.

**Se mudou a referência do monitoramento**
- Grave a nova referência como artefato da run do champion, em `referencia/`, com um nome que diga o que ela é e de que período.

**Sempre**
- Uma entrada no `CHANGELOG.md` por versão nova, de modelo, pipeline ou contrato. Use o modelo de entrada do próprio arquivo.
- Mostre o **antes e depois** nas semanas afetadas: custo, bloqueio e recall.

### Ações concluídas

- Aplicamos a correção de entrada v1.1.1 aos lotes 15 e 16 usando `corrigir_lote_vertice.py`. Os arquivos de origem e os scores originais foram preservados. A semana 14 ficou com a pontuação histórica.
- Conferimos cinco scores e decisões por lote no endpoint `/predict`. O MLflow manteve `antifraude v7` como `@champion`, corte de 0,09 e contrato v1.0.
- As runs de auditoria são `8b3dff66abf04f80896908490dc00f8d` (semana 15) e `b34d20d45413499e82cb14af7cfd5a83` (semana 16). Os relatórios antes e depois da correção e as quatro decisões estão no ZIP de entrega.

**Escopo da correção.** A mudança foi no dado dos dois lotes recebidos da Vértice. O artefato de pontuação, o corte, o contrato e a referência continuaram identificados pela mesma v7. A correção de entrada recebeu a versão v1.1.1 no changelog e foi ligada às duas runs de auditoria. A semana 13 já estava coerente, e a semana 14 foi preservada como histórico para medir o impacto com as contestações. A comparação com a semana 14 corrigida é retrospectiva.

In [19]:
import hashlib
import json

evidencias_correcao = {}
for semana in (15, 16):
    pasta = pasta_relatorios / f"semana_{semana}" / "apos_correcao"
    evidencia = json.loads((pasta / "correcao_horario.json").read_text(encoding="utf-8"))
    auditoria = json.loads((pasta / "run_mlflow.json").read_text(encoding="utf-8"))
    assert evidencia["semana"] == semana and evidencia["versao_correcao"] == "1.1.1"
    assert evidencia["antes"]["transacoes"] == evidencia["depois"]["transacoes"]
    for chave in ("predicoes_originais", "predicoes_corrigidas"):
        primeiro, ultimo = evidencia[chave]
        assert ultimo - primeiro + 1 == evidencia["depois"]["transacoes"]
    assert evidencia["predicoes_originais"][1] < evidencia["predicoes_corrigidas"][0]
    for arquivo, chave in (
        (Path("dados/parte2") / evidencia["arquivo"], "sha256_original"),
        (pasta / f"transacoes_semana_{semana}_corrigidas.csv", "sha256_corrigido"),
    ):
        if arquivo.is_file():
            assert hashlib.sha256(arquivo.read_bytes()).hexdigest() == evidencia[chave]
    evidencias_correcao[semana] = (evidencia, auditoria)

display(pd.DataFrame([
    {"semana": semana, "correcao": ev["versao_correcao"], "origem": ev["origem"],
     "destino": ev["destino"], "predicoes_originais": ev["predicoes_originais"],
     "predicoes_corrigidas": ev["predicoes_corrigidas"]}
    for semana, (ev, _) in evidencias_correcao.items()
]))

,semana,correcao,origem,destino,predicoes_originais,predicoes_corrigidas
0,15,1.1.1,vertice-utc,America/Sao_Paulo,"[39903, 59964]","[59965, 80026]"
1,16,1.1.1,vertice-utc,America/Sao_Paulo,"[80027, 99985]","[99986, 119944]"


In [20]:
import requests

from app.contrato import nomes_features
from monitoramento import carregar_semana

verificacoes_api = []
for semana in (15, 16):
    vigente = carregar_semana(semana)
    amostra = vigente.loc[vigente["hora_transacao"].ge(21)].head(5)
    assert len(amostra) == 5
    entrada = json.loads(amostra[nomes_features()].to_json(orient="records"))
    resposta = requests.post(config.SISTEMA_URL.rstrip("/") + "/predict", json=entrada, timeout=30)
    resposta.raise_for_status()
    predicoes = pd.DataFrame(resposta.json())
    assert predicoes["versao_modelo"].astype(str).eq("7").all() and predicoes["threshold"].eq(0.09).all()
    diferenca = (predicoes["score"] - amostra["score"].reset_index(drop=True)).abs().max()
    assert diferenca <= 5.01e-7
    assert predicoes["bloquear"].tolist() == amostra["score"].ge(0.09).tolist()
    verificacoes_api.append({"semana": semana, "scores_conferidos": len(predicoes),
                             "diferenca_maxima_por_arredondamento": diferenca,
                             "decisoes_iguais": True})
display(pd.DataFrame(verificacoes_api))

,semana,scores_conferidos,diferenca_maxima_por_arredondamento,decisoes_iguais
0,15,5,4.713779e-07,True
1,16,5,3.276249e-07,True


In [21]:
champion_final = cli.get_model_version_by_alias(config.NOME_MODELO, config.ALIAS_PRODUCAO)
assert str(champion_final.version) == "7"
assert float(champion_final.tags[config.TAG_THRESHOLD]) == 0.09
assert champion_final.tags.get(config.TAG_MOTIVO) and champion_final.description
referencia_final = {a.path for a in cli.list_artifacts(champion_final.run_id, "referencia")}
assert {"referencia/metadados.json", "referencia/validacao_11_12.csv"} <= referencia_final

rastreabilidade = []
for semana, (evidencia, auditoria) in evidencias_correcao.items():
    run = cli.get_run(auditoria["run_id"])
    assert run.info.status == "FINISHED"
    assert run.data.params["run_modelo"] == champion_final.run_id == evidencia["modelo"]["run_id"]
    arquivos = {a.path for a in cli.list_artifacts(run.info.run_id, "correcao_horario")}
    assert "correcao_horario/correcao_horario.json" in arquivos
    assert run.info.run_id in Path("CHANGELOG.md").read_text(encoding="utf-8")
    rastreabilidade.append({"semana": semana, "correcao": evidencia["versao_correcao"],
                            "contrato": evidencia["contrato"], "champion": champion_final.version,
                            "run_modelo": champion_final.run_id, "run_correcao": run.info.run_id})
display(pd.DataFrame(rastreabilidade))

,semana,correcao,contrato,champion,run_modelo,run_correcao
0,15,1.1.1,1.0,7,98e2f8417abf497ebb1132c0a39f5258,8b3dff66abf04f80896908490dc00f8d
1,16,1.1.1,1.0,7,98e2f8417abf497ebb1132c0a39f5258,b34d20d45413499e82cb14af7cfd5a83


In [22]:
comparacao = pd.read_csv(pasta_relatorios / "semana_16" / "comparacao_horario.csv")
linhas = comparacao.loc[comparacao["semana"].isin([14, 15, 16]), [
    "semana", "cenario", "retidas", "taxa_retencao", "custo", "recall", "clientes_legitimos_afetados"
]].copy()
linhas["uso"] = ["histórico", "simulação", "pontuação original", "após correção", "pontuação original", "após correção"]
assert linhas.loc[linhas["semana"].eq(16), ["custo", "recall"]].isna().all().all()
print("A semana 16 aguarda contestações. As colunas de custo e recall dessa semana ficam em branco.")
display(linhas[["semana", "uso", "retidas", "taxa_retencao", "custo", "recall", "clientes_legitimos_afetados"]])

A semana 16 aguarda contestações. As colunas de custo e recall dessa semana ficam em branco.


,semana,uso,retidas,taxa_retencao,custo,recall,clientes_legitimos_afetados
1,14,histórico,812,0.039643,15065.62,0.841584,553.0
2,14,simulação,553,0.026998,10525.93,0.858086,291.0
3,15,pontuação original,750,0.037384,14503.03,0.781377,553.0
4,15,após correção,506,0.025222,9759.94,0.813765,303.0
5,16,pontuação original,803,0.040232,NaN,NaN,NaN
6,16,após correção,546,0.027356,NaN,NaN,NaN


## Comunicação

Escreva a resposta para a Helena no arquivo `resposta_parte2.md`. Ela vai levar essa resposta para a diretoria. Responda também às perguntas que apareceram nos anexos.

Teste rápido antes de entregar: a Helena consegue repetir a sua explicação para o diretor, em uma frase, sem usar nenhuma palavra que ela não conheça?

A resposta à Helena está em `resposta_parte2.md`.

Frase para a diretoria: a migração adiantou os horários registrados e levou compras legítimas à verificação. Corrigimos o horário e, na semana 15, as contestações indicaram R$ 4.743,09 a menos de custo em relação à pontuação original.

## Entrega

1. Confira a checklist:
   - [x] as semanas 13 a 16 estão no sistema, e os rótulos de 13 a 15 também;
   - [x] há uma decisão registrada por semana, de 13 a 16;
   - [x] o `@champion` tem as tags `threshold` e `motivo`, uma descrição e a `referencia/` na run;
   - [x] o `CHANGELOG.md` cita a versão que está em produção e, se o contrato mudou, a versão nova do contrato;
   - [x] o `resposta_parte2.md` está escrito;
   - [x] as células de texto deste notebook estão respondidas.
2. Preencha o seu nome, **salve o notebook** e rode a célula abaixo. Ela gera a pasta `entrega_parte2/` e o `entrega_parte2_<nome>.zip`. A pasta leva as evidências do MLflow, o modelo exportado, a referência, o painel, as decisões e os seus arquivos.
3. Entregue o `.zip` na atividade. Se a célula listar **avisos**, resolva antes de entregar.

> A célula não mexe em nada no MLflow nem no sistema: ela só lê e copia.

In [1]:
# CÉLULA PRONTA — gera a pasta entrega_parte2/ e o entrega_parte2_<nome>.zip.
# 1) Preencha o seu nome.  2) SALVE o notebook (Ctrl+S).  3) Rode.
NOME_ALUNO = "Levy Guilherme Lemos Rios"

import datetime
import json
import re
import shutil
import tempfile
from pathlib import Path

import io

import mlflow
import pandas as pd
import requests
import yaml
from mlflow.tracking import MlflowClient

import config


def _api(rota):
    r = requests.get(config.SISTEMA_URL.rstrip("/") + rota, timeout=120)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text)) if rota.endswith(".csv") else r.json()


SEMANAS = [13, 14, 15, 16]
ROTULOS = [13, 14, 15]

assert NOME_ALUNO.strip(), "Preencha NOME_ALUNO antes de rodar."
mlflow.set_tracking_uri(config.MLFLOW_TRACKING_URI)
cli = MlflowClient()
mv = cli.get_model_version_by_alias(config.NOME_MODELO, config.ALIAS_PRODUCAO)
versoes = sorted(cli.search_model_versions(f"name='{config.NOME_MODELO}'"), key=lambda v: int(v.version))
art_ref = [a.path for a in cli.list_artifacts(mv.run_id, "referencia")]
sist = _api("/api/modelo")
painel = _api("/api/painel.csv")
lotes = pd.DataFrame(_api("/api/lotes"))
decisoes = pd.DataFrame(_api("/api/decisoes"))
contrato = yaml.safe_load(Path(config.CONTRATO).read_text(encoding="utf-8"))
changelog = Path("CHANGELOG.md").read_text(encoding="utf-8")
resposta = Path("resposta_parte2.md").read_text(encoding="utf-8")

avisos = []
recebidas = set(lotes["semana"]) if len(lotes) else set()
for s in SEMANAS:
    if s not in recebidas:
        avisos.append(f"a semana {s} não foi enviada ao sistema")
for s in ROTULOS:
    linha = lotes[lotes["semana"] == s] if len(lotes) else lotes
    if len(linha) and int(linha["rotulos"].iloc[0]) != int(linha["qtd_linhas"].iloc[0]):
        avisos.append(f"a cobertura de rótulos da semana {s} está incompleta")
for s in SEMANAS:
    if int(decisoes["semana"].eq(s).sum()) != 1:
        avisos.append(f"a semana {s} precisa ter exatamente uma decisão registrada")
    linha_painel = painel.loc[painel["semana"].eq(s)]
    if linha_painel.empty or int(linha_painel["pontuadas"].iloc[0]) != int(linha_painel["volume"].iloc[0]):
        avisos.append(f"o lote da semana {s} não está totalmente pontuado")
if lotes.loc[lotes["semana"].eq(16), "rotulos"].gt(0).any():
    avisos.append("a semana 16 deve aguardar as contestações da Parte 3")
for chave in (config.TAG_THRESHOLD, config.TAG_MOTIVO):
    if chave not in mv.tags:
        avisos.append(f"a versão champion (v{mv.version}) não tem a tag '{chave}'")
if not mv.description:
    avisos.append(f"a versão champion (v{mv.version}) não tem descrição")
if not art_ref:
    avisos.append("a run do champion não tem artefatos em 'referencia/'")
if str(sist["modelo"].get("versao")) != str(mv.version):
    avisos.append("o sistema não está usando a mesma versão que o MLflow aponta como champion")
if not re.search(rf"{config.NOME_MODELO}\s+v{mv.version}\b", changelog):
    avisos.append(f"o CHANGELOG.md não cita a versão em produção ('{config.NOME_MODELO} v{mv.version}')")
if str(contrato["versao"]) != "1.0" and str(contrato["versao"]) not in changelog:
    avisos.append(f"o contrato está na versão {contrato['versao']}, mas o CHANGELOG.md não cita essa versão")
texto_resposta = re.sub(r"<!--.*?-->", "", resposta, flags=re.S)
if len(texto_resposta.split()) < 80:
    avisos.append("resposta_parte2.md parece vazia ou curta demais")

correcoes = {}
for semana in (15, 16):
    pasta_auditoria = Path(f"monitoramento/semana_{semana}/apos_correcao")
    registro = json.loads((pasta_auditoria / "run_mlflow.json").read_text(encoding="utf-8"))
    corrida = cli.get_run(registro["run_id"])
    artefatos = [a.path for a in cli.list_artifacts(corrida.info.run_id, "correcao_horario")]
    if corrida.info.status != "FINISHED" or "correcao_horario/correcao_horario.json" not in artefatos:
        avisos.append(f"auditoria do lote {semana} incompleta no MLflow")
    correcoes[str(semana)] = {**registro, "artefatos": artefatos}

assert not avisos, "\n".join(avisos)
evidencias = {
    "aluno": NOME_ALUNO.strip(),
    "gerado_em": datetime.datetime.now().isoformat(timespec="seconds"),
    "champion": {"versao": mv.version, "run_id": mv.run_id, "descricao": mv.description, "tags": dict(mv.tags)},
    "versoes": [{"versao": v.version, "aliases": list(v.aliases), "run_id": v.run_id, "criada_em": v.creation_timestamp,
                 "descricao": v.description, "tags": dict(v.tags)} for v in versoes],
    "referencia_champion": art_ref,
    "contrato": {"versao": contrato["versao"], "regras": contrato.get("regras") or [],
                 "changelog": contrato.get("changelog") or []},
    "sistema": sist,
    "correcoes_horario": correcoes,
    "avisos": avisos,
}

pasta = Path("entrega_parte2")
pasta.mkdir(exist_ok=True)
(pasta / "evidencias_mlflow.json").write_text(json.dumps(evidencias, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
painel.to_csv(pasta / "painel.csv", index=False)
decisoes.to_csv(pasta / "decisoes.csv", index=False)
lotes.to_csv(pasta / "lotes.csv", index=False)

with tempfile.TemporaryDirectory(dir=pasta) as temporaria:
    try:
        origem_modelo = Path(mlflow.artifacts.download_artifacts(
            artifact_uri=f"models:/{config.NOME_MODELO}/{mv.version}", dst_path=str(Path(temporaria).resolve())))
    except (mlflow.exceptions.MlflowException, FileNotFoundError):
        # A exportação da Parte 1 serve se corresponder ao champion verificado.
        origem_modelo = Path("entrega/modelo_exportado")
        registro_modelo = yaml.safe_load((origem_modelo / "registered_model_meta").read_text(encoding="utf-8"))
        assert registro_modelo["model_name"] == config.NOME_MODELO and str(registro_modelo["model_version"]) == str(mv.version)
    manifesto = yaml.safe_load((origem_modelo / "MLmodel").read_text(encoding="utf-8"))
    assert manifesto["run_id"] == mv.run_id
    shutil.copytree(origem_modelo, pasta / "modelo_exportado", dirs_exist_ok=True)
if art_ref:
    mlflow.artifacts.download_artifacts(run_id=mv.run_id, artifact_path="referencia", dst_path=str(pasta.resolve()))

for arq in ("parte2.ipynb", "features.py", "monitoramento.py", "CHANGELOG.md", "resposta_parte2.md",
            "corrigir_lote_vertice.py", "avaliar_modelos.py", "config.py", "app/__init__.py",
            "app/banco.py", "app/contrato.py", "contrato/contrato_features.yaml"):
    destino = pasta / arq
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(arq, destino)
shutil.copy(config.CONTRATO, pasta / "contrato_features.yaml")

for semana in SEMANAS:
    raiz_relatorio = Path(f"monitoramento/semana_{semana}")
    destino = pasta / "evidencias_operacao" / f"semana_{semana}"
    destino.mkdir(parents=True, exist_ok=True)
    for arquivo in raiz_relatorio.glob("*.csv"):
        shutil.copy(arquivo, destino)
    if semana in (15, 16):
        apos = destino / "apos_correcao"
        apos.mkdir(exist_ok=True)
        for nome in ("continuas.csv", "discretas.csv", "saida.csv", "performance.csv",
                     "correcao_horario.json", "run_mlflow.json"):
            shutil.copy(raiz_relatorio / "apos_correcao" / nome, apos)
shutil.copy("monitoramento/semana_16/totais_confirmados.json", pasta / "evidencias_operacao")

nome_zip = "entrega_parte2_" + "_".join(NOME_ALUNO.lower().split())
shutil.make_archive(nome_zip, "zip", pasta)
print(f"champion: {config.NOME_MODELO} v{mv.version} | threshold {mv.tags.get(config.TAG_THRESHOLD)} | contrato v{contrato['versao']}")
print(f"decisões registradas: {len(decisoes)} | trocas de modelo no sistema: {len(sist['trocas'])}")
print("AVISOS:" if avisos else "Nenhum aviso.", *avisos, sep="\n  - ")
print(f"\nEntregue o arquivo {nome_zip}.zip")

c:\Users\levyrios-ieg\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


champion: antifraude v7 | threshold 0.09 | contrato v1.0
decisões registradas: 4 | trocas de modelo no sistema: 5
Nenhum aviso.

Entregue o arquivo entrega_parte2_levy_guilherme_lemos_rios.zip
